# MIMIC-IV Chart Events Exploration

Dedicated exploratory analysis of the `chartevents` table (vitals and nursing observations). This notebook computes top item frequencies using chunked processing to handle the large table size.

## Section 1: Setup & Configuration

In [29]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import defaultdict, Counter

# Set plot style for better readability
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Define file paths for chart events
base_path = '~/data/physionet.org/files/mimiciv/3.1/'
chartevents_file = base_path + 'icu/chartevents.csv.gz'
d_items_file = base_path + 'icu/d_items.csv.gz'
print("File paths set for chartevents and d_items.")


## Section 2: Chart Events (Vitals) Exploration

Analyze chart events which contain vital signs and nursing observations. This is the largest table in MIMIC-IV.

In [ ]:
# Load d_items dictionary for mapping itemid to label
print("Loading d_items dictionary...")
d_items_df = pd.read_csv(d_items_file)
print(f"Total chart items in dictionary: {len(d_items_df)}")

# Create mapping from itemid to label
chartitem_map = dict(zip(d_items_df['itemid'], d_items_df['label']))
print(f"Mapping created for {len(chartitem_map)} chart items.")

In [ ]:
# Read chartevents in chunks and count itemid frequency
print("\nReading chartevents file in chunks (this may take a while)...")
chunk_size = 1000000
chart_itemid_frequency = defaultdict(int)
chunk_number = 0

for chunk in pd.read_csv(
    chartevents_file,
    usecols=['itemid'],
    chunksize=chunk_size
):
    chunk_number += 1
    print(f"Processing chunk {chunk_number}... ({len(chunk):,} rows)")
    
    # Count frequency of each itemid
    itemid_counts = chunk['itemid'].value_counts()
    for itemid, count in itemid_counts.items():
        chart_itemid_frequency[itemid] += count

print(f"\nTotal chunks processed: {chunk_number}")
print(f"Unique chart event types: {len(chart_itemid_frequency):,}")

In [ ]:
# Get top 20 most frequent chart events
top_20_chartitems = sorted(chart_itemid_frequency.items(), key=lambda x: x[1], reverse=True)[:20]

# Map itemids to labels
top_20_chart_labels = []
top_20_chart_counts = []
for itemid, count in top_20_chartitems:
    label = chartitem_map.get(itemid, f"Unknown ({itemid})")
    top_20_chart_labels.append(label)
    top_20_chart_counts.append(count)

print("\nTOP 20 MOST FREQUENT CHART EVENTS (VITALS/OBSERVATIONS):")
for i, (label, count) in enumerate(zip(top_20_chart_labels, top_20_chart_counts), 1):
    print(f"{i:2d}. {label:50s} - {count:,} occurrences")

In [ ]:
# Visualization: Horizontal bar chart of top 20 chart events
plt.figure(figsize=(12, 8))
y_pos = np.arange(len(top_20_chart_labels))
plt.barh(y_pos, top_20_chart_counts, alpha=0.7, color='steelblue')
plt.yticks(y_pos, top_20_chart_labels)
plt.xlabel('Number of Occurrences')
plt.ylabel('Chart Event (Vital/Observation)')
plt.title('Top 20 Most Frequent Chart Events')
plt.gca().invert_yaxis()  # Highest at the top
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()